In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
import os
from malware_pck import to_wav as wav
import importlib
importlib.reload(wav)

import librosa
import librosa.display
from tqdm import tqdm

<module 'malware_pck.to_wav' from '/home/tdfink/malware_project/src/malware_pck/to_wav.py'>

In [ ]:
saved_indices = wav.create_noUnknown_audios(100000)
wav.show_samples(saved_indices)

Generating exactly 100000 labeled audio files...


100%|█████████████████████████████████████████████████████████████████████████| 100000/100000 [00:23<00:00, 4324.68it/s]


Generation complete.
Out of the 100000 audio files generated from X_train:
Total Malware: 50002
Total Benign: 49998
Total Unlabeled: 0


In [ ]:
# Convert WAV files → Mel Spectrogram PNGs
SAMPLE_RATE  = 8000   # must match what to_wav.py used
N_MELS       = 128
N_TILES      = 10     # tile the short signal to get more time frames
WAV_DIR      = "../data/audio"
SPEC_DIR     = "../data/spectrograms"

def wav_to_spectrogram(wav_path, out_path):
    """Load a WAV, tile it to extend duration, convert to Mel spectrogram PNG."""
    y, sr = librosa.load(wav_path, sr=SAMPLE_RATE, mono=True)

    # Tile the signal to artificially extend it (fixes the short signal problem)
    y = np.tile(y, N_TILES)

    # Compute Mel spectrogram
    mel      = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS)
    mel_db   = librosa.power_to_db(mel, ref=np.max)

    # Render to a clean 128x128 grayscale PNG (no axes, no borders)
    fig, ax = plt.subplots(figsize=(1.28, 1.28), dpi=100)
    librosa.display.specshow(mel_db, sr=sr, ax=ax, cmap='gray')
    ax.axis('off')
    fig.subplots_adjust(left=0, right=1, top=1, bottom=0)
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    fig.savefig(out_path, dpi=100, bbox_inches='tight', pad_inches=0)
    plt.close(fig)


def convert_all_wavs_to_spectrograms():
    """Walk the audio folder structure and convert every WAV to a spectrogram PNG."""
    for class_name in ['benign', 'malware']:
        wav_class_dir  = os.path.join(WAV_DIR,  class_name)
        spec_class_dir = os.path.join(SPEC_DIR, class_name)
        os.makedirs(spec_class_dir, exist_ok=True)

        wav_files = [f for f in os.listdir(wav_class_dir) if f.endswith('.wav')]
        print(f"Converting {len(wav_files)} {class_name} WAV files to spectrograms...")

        for fname in tqdm(wav_files):
            wav_path  = os.path.join(wav_class_dir,  fname)
            spec_path = os.path.join(spec_class_dir, fname.replace('.wav', '.png'))

            # Skip if already converted (allows safe resuming)
            if os.path.exists(spec_path):
                continue

            try:
                wav_to_spectrogram(wav_path, spec_path)
            except Exception as e:
                print(f"  [WARNING] Failed on {fname}: {e}")

    print("Spectrogram conversion complete.")

convert_all_wavs_to_spectrograms()

In [ ]:
# Load spectrogram PNGs as image datasets
IMG_HEIGHT = 128
IMG_WIDTH  = 128
BATCH_SIZE = 32
data_dir   = "../data/spectrograms"

In [ ]:
print("Loading Training Data:")
train_ds = tf.keras.utils.image_dataset_from_directory(
  data_dir,
  validation_split=0.2, 
  subset="training",
  seed=42,
  image_size=(IMG_HEIGHT, IMG_WIDTH),
  batch_size=BATCH_SIZE,
  color_mode="grayscale" # Grayscale is perfect for feature vectors
)

In [ ]:
print("Loading Validation Data:")
val_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    color_mode="grayscale"
)

In [ ]:
# Optimize the loading speed for both datasets
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds   = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

In [ ]:
print("Defining Baseline Audio Model architecture...")

# Create a Sequential model
audio_model = models.Sequential([
    # Input layer and feature rescaling (important for stability)
    layers.Rescaling(1./255, input_shape=(IMG_HEIGHT, IMG_WIDTH, 1)),

    # First convolutional block
    layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
    layers.MaxPooling2D(),

    # Second convolutional block
    layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
    layers.MaxPooling2D(),

    # Transition to dense layers
    layers.Flatten(),
    layers.Dense(128, activation='relu'),

    # Output layer for binary classification (Malware vs Benign)
    layers.Dense(1, activation='sigmoid')
])

print("Model architecture defined.")
audio_model.summary()

In [ ]:
# Compile
print("Compiling the model...")
audio_model.compile(optimizer='adam',
                    loss='binary_crossentropy',
                    metrics=['accuracy'])
print("Model compiled.")

In [ ]:
# Train with class weighting
EPOCHS = 10

# Define weights: Class 1 (Malware) is 5x more important than Class 0 (Benign)
class_weights = {0: 1.0, 1: 1.5}

print("Starting audio model training with malware weighting...")

history = audio_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    class_weight=class_weights
)
print("Model training complete.")

In [ ]:
# Extract accuracy and loss metrics from the history object
# Training curves
acc          = history.history['accuracy']
val_acc      = history.history['val_accuracy']
loss         = history.history['loss']
val_loss     = history.history['val_loss']

epochs_range = range(len(acc))

# Set up a wide canvas for two side-by-side plots
plt.figure(figsize=(14, 5))

# 1. Plot Training and Validation Accuracy
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc,     label='Training Accuracy',   color='blue',   linewidth=2)
plt.plot(epochs_range, val_acc, label='Validation Accuracy', color='orange', linewidth=2, linestyle='--')
plt.xlabel('Epochs'); plt.ylabel('Accuracy')
plt.title('Audio Model – Training and Validation Accuracy')
plt.legend(loc='lower right'); plt.grid(True, linestyle=':', alpha=0.6)

# 2. Plot Training and Validation Loss
plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss,     label='Training Loss',   color='blue',   linewidth=2)
plt.plot(epochs_range, val_loss, label='Validation Loss', color='orange', linewidth=2, linestyle='--')
plt.xlabel('Epochs'); plt.ylabel('Loss')
plt.title('Audio Model – Training and Validation Loss')
plt.legend(loc='upper right'); plt.grid(True, linestyle=':', alpha=0.6)

plt.tight_layout()
plt.show()

In [ ]:
# Evaluation with adjustable threshold
print("Generating predictions on the validation set...")
y_pred_probs = audio_model.predict(val_ds)

custom_threshold = 0.2
y_pred = (y_pred_probs > custom_threshold).astype(int).flatten()

y_true = []
for _, labels in val_ds:
    y_true.extend(labels.numpy())
y_true = np.array(y_true)

cm = confusion_matrix(y_true, y_pred)
tn, fp, fn, tp = cm.ravel()

print("\n" + "="*45)
print(f"   RESULTS AT THRESHOLD: {custom_threshold}")
print("="*45)
print(f"True Negatives  (Benign correctly identified):  {tn}")
print(f"True Positives  (Malware correctly identified): {tp}")
print(f"False Positives (Benign flagged as Malware):    {fp}")
print(f"False Negatives (Malware missed!):              {fn}")
print("="*45)

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Benign', 'Malware'])
disp.plot(cmap='Blues', values_format='d')
plt.title(f'Audio Model – Confusion Matrix (Threshold: {custom_threshold})')
plt.show()